In [2]:
import pandas as pd

data = [
    ['adaptive_rbf', 0.9, 0.1406, 0.1592, 1.8338, 0.6324, 0.4958, 0.1311],
    ['adaptive_rbf', 1.0, 0.1285, 0.1606, 2.0322, 0.6284, 0.5247, 0.0908],
    ['adaptive_rbf', 1.1, 0.1173, 0.1490, 1.9606, 0.5923, 0.5033, 0.0960],
    ['cosine', 0.9, 0.1537, 0.1612, 2.0319, 0.6637, 0.5765, 0.3081],
    ['cosine', 1.0, 0.1462, 0.1682, 2.1402, 0.7072, 0.6169, 0.2822],
    ['cosine', 1.1, 0.1469, 0.1674, 2.1602, 0.6850, 0.6097, 0.2902],
    ['mutual_knn', 0.9, 0.1477, 0.1651, 2.0158, 0.6569, 0.5446, 0.2339],
    ['mutual_knn', 1.0, 0.1348, 0.1675, 2.1716, 0.6734, 0.5907, 0.2120],
    ['mutual_knn', 1.1, 0.1263, 0.1622, 2.1602, 0.6433, 0.5755, 0.2174],
    ['tanimoto', 0.9, 0.1516, 0.0905, 0.8914, 0.6304, 0.2576, 0.3804],
    ['tanimoto', 1.0, 0.1426, 0.0958, 0.9234, 0.6515, 0.2741, 0.3590],
    ['tanimoto', 1.1, 0.0201, 0.0819, 1.0023, 0.5065, 0.2521, 0.3366],
]

columns = ['method', 'resolution', 'SW', 'CH', 'S_Dbw', 'AVI', 'AVU', 'MQ']

results_df = pd.DataFrame(data, columns=columns)

metrics = ['SW', 'CH', 'S_Dbw', 'AVI', 'AVU', 'MQ']

# Spearman correlation
corr_spearman = results_df[metrics].corr(method='spearman')

corr_spearman.round(3)


,SW,CH,S_Dbw,AVI,AVU,MQ
SW,1.000,0.266,-0.077,0.587,0.294,0.483
CH,0.266,1.000,0.869,0.846,0.958,-0.322
S_Dbw,-0.077,0.869,1.000,0.599,0.879,-0.466
AVI,0.587,0.846,0.599,1.000,0.846,0.147
AVU,0.294,0.958,0.879,0.846,1.000,-0.259
MQ,0.483,-0.322,-0.466,0.147,-0.259,1.000


Видим, что значения CH, S_Dbw, AVI, AVU коррелируют, поэтому вместо обычного Borda Count будем использовать взвешенный.

In [3]:
import pandas as pd
import numpy as np

metrics = [
    'SW',
    'CH',
    'S_Dbw',
    'AVI',
    'AVU',
    'MQ'
]

# Задаём направление метрикам:
# True  -> больше = лучше
# False -> меньше = лучше
higher_is_better = {
    'SW': True,
    'CH': True,
    'S_Dbw': False,
    'AVI': True,
    'AVU': False,
    'MQ': True
}

In [4]:
norm_df = pd.DataFrame(index=results_df.index)

for metric in metrics:
    x = results_df[metric]

    if higher_is_better[metric]:
        norm_df[metric] = (x - x.min()) / (x.max() - x.min())
    else:
        norm_df[metric] = (x.max() - x) / (x.max() - x.min())

In [5]:
std = norm_df.std()

corr = norm_df.corr(method='pearson')

In [6]:
critic_score = {}

for metric in metrics:
    conflict = sum(
        1 - corr.loc[metric, other]
        for other in metrics
        if other != metric
    )

    critic_score[metric] = std[metric] * conflict

critic_score = pd.Series(critic_score)

weights = critic_score / critic_score.sum()

print("Веса метрик:")
print(weights.sort_values(ascending=False).round(3))

Веса метрик:
CH       0.232
AVU      0.201
S_Dbw    0.199
MQ       0.144
SW       0.114
AVI      0.110
dtype: float64


In [7]:
borda = pd.DataFrame(index=results_df.index)

n = len(results_df)

for metric in metrics:

    if higher_is_better[metric]:
        rank = results_df[metric].rank(
            ascending=False,
            method='average'
        )
    else:
        rank = results_df[metric].rank(
            ascending=True,
            method='average'
        )

    # Borda: 1 место получает максимум баллов
    borda[metric] = n - rank

In [8]:
weighted_borda = pd.Series(0.0, index=results_df.index)

for metric in metrics:
    weighted_borda += borda[metric] * weights[metric]

In [9]:
ranking = results_df[
    ['method', 'resolution']
].copy()

ranking['Borda_score'] = weighted_borda

ranking = ranking.sort_values(
    'Borda_score',
    ascending=False
).reset_index(drop=True)

ranking

,method,resolution,Borda_score
0,tanimoto,0.9,7.477259
1,tanimoto,1.0,7.042713
2,mutual_knn,0.9,6.571999
3,cosine,0.9,6.276786
4,cosine,1.0,6.028179
5,cosine,1.1,5.613125
6,adaptive_rbf,0.9,5.423303
7,tanimoto,1.1,5.291390
8,mutual_knn,1.0,4.606982
9,mutual_knn,1.1,4.083672
